# Cohort Creation Workflow

Orchestrates Step 2 cohort creation with **one cell per cohort**. Run the configuration cell once, then run the cell for the cohort series you want.

**Important:** The underlying pipeline (`0_create_cohort.py`) **builds and writes both cohorts** for every (age_band, event_year) partition. So when you run the **first** cohort cell (OPIOID_ED), it creates **both** OPIOID_ED and POLYPHARMACY (ed_non_opioid) outputs. Running the second cell (POLYPHARMACY) with `--skip-existing` then skips all partitions because they already exist.

- **Cohort 1 cell** — Run this to build all partitions; you get both cohorts.
- **Cohort 2 cell** — Use when you want to run only the polypharmacy series (e.g. without `--skip-existing` to force rebuild, or on another run).

## Cohorts

1. **OPIOID_ED** (`opioid_ed`) — F11.20 target; `run_series_opioid_ed.py` (writes both cohorts per partition)
2. **POLYPHARMACY** (`ed_non_opioid`) — time-windowed HCG target; `run_series_ed_non_opioid.py`

## Prerequisites

- Step 1a: APCD input data in gold (medical/pharmacy)
- Optional: Step 1b event filter run with `--before-cohorts` (filtered gold used when present)

## Configuration

In [ ]:
import sys
from pathlib import Path

# Project root (notebook at repo root)
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PYTHON_BIN = Path(sys.executable)
LOG_DIR = PROJECT_ROOT / "2_create_cohort" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {PYTHON_BIN}")
print(f"Log dir: {LOG_DIR}")

## Cohort 1: OPIOID_ED

Runs `run_series_opioid_ed.py` for all age bands and event years (skip-existing by default).

In [ ]:
import subprocess

script = PROJECT_ROOT / "2_create_cohort" / "run_series_opioid_ed.py"
cmd = [str(PYTHON_BIN), str(script), "--skip-existing", "--concurrent-workers", "1"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)

## Cohort 2: POLYPHARMACY (ed_non_opioid)

Runs `run_series_ed_non_opioid.py` for all age bands and event years. Use this cell if you want to run only the polypharmacy series (e.g. without `--skip-existing` to force rebuild). If you already ran Cohort 1, partitions will be skipped because both cohorts were written then.

In [ ]:
import subprocess

script = PROJECT_ROOT / "2_create_cohort" / "run_series_ed_non_opioid.py"
cmd = [str(PYTHON_BIN), str(script), "--skip-existing", "--concurrent-workers", "1"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)

## Check S3 completion status (with time durations)

Pipeline state is stored in **pgx-repository** (`pgx-pipeline-status/create_cohort/`). Each partition has a `state.json` with `created_at`, `completed_at`, and `status`. Run the cell below to list completion status and **duration** (completed_at − created_at). Optionally list cohort parquets in **pgxdatalake** with LastModified and size (`--outputs`).

In [ ]:
import subprocess
script = PROJECT_ROOT / "2_create_cohort" / "check_s3_cohort_completion.py"
# Use --outputs to list each cohort.parquet with LastModified and size
subprocess.run([str(PYTHON_BIN), str(script), "--outputs"], cwd=str(PROJECT_ROOT))